<a href="https://colab.research.google.com/github/Werianin/shift-MLI-2026/blob/main/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gradio as gr
import joblib
import pandas as pd

In [2]:
model = joblib.load('/rf_model.joblib')
le = joblib.load('/label_encoders.joblib')

In [3]:
features=['Пол',
          'Город',
          'Образование',
          'Курсы',
          'ГодНайма',
          'Оценка_HR',
          'ОпытВДолжности',
          'УровеньОплаты',
          'Закрытые_проекты',
          'Активность',
          'Офлайн_участие'
          ]
target='Увольнение'
target_names = ['Не уволится', 'Уволится']

In [4]:
def predict_exit(sex, city, education, courses, year_of_hire, hr_rating,
                 experience, pay_level, closed_projects, activity, offline_p):
    # СОЗДАЕМ DATAFRAME ИЗ ВХОДНЫХ ДАННЫХ
    input_data = pd.DataFrame([[sex, city, education, courses, year_of_hire,
                                hr_rating, experience, pay_level,
                                closed_projects, activity, offline_p]],
                              columns=features)

    # ГРУППИРОВАНИЕ ГОРОДОВ
    input_data['Город'] = input_data['Город'].replace({'Bangalore': 'Big_City',
                                                     'New Delhi': 'Big_City'})

    # ПОРЯДКОВОЕ КОДИРОВАНИЕ ОБРАЗОВАНИЯ
    edu_map = {'PHD': 0, 'Bachelors': 1, 'Masters': 2}
    input_data['Образование'] = input_data['Образование'].map(edu_map)

    # ПРИМЕНЕНИЕ LABEL ENCODER ДЛЯ КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ
    input_data['Пол'] = le['Пол'].transform(input_data['Пол'])
    input_data['Город'] = le['Город'].transform(input_data['Город'])

    # ПРЕДСКАЗАНИЕ
    prediction = model.predict(input_data)[0]
    result = 'Уволится' if prediction else 'Не уволится'

    # ВЕРОЯТНОСТИ
    proba = model.predict_proba(input_data)[:,prediction][0]

    return f"Вид: {result}\n Уверенность: {proba}"



In [5]:
# Описание входных полей
inputs = [
    gr.Dropdown(choices=['Male', 'Female'], label="Пол"),
    gr.Dropdown(choices=['Pune', 'New Delhi', 'Bangalore'], label="Город"),
    gr.Dropdown(choices=['Bachelors', 'Masters', 'PHD'], label="Образование"),
    gr.Slider(0, 5, step=1, label="Курсы"),
    gr.Slider(2012, 2018, step=1, label="ГодНайма"),
    gr.Slider(0, 10, step=1, label="Оценка_HR"),
    gr.Slider(0, 7, step=1, label="ОпытВДолжности"),
    gr.Slider(1, 3, step=1, label="УровеньОплаты"),
    gr.Slider(0, 15, step=1, label="Закрытые_проекты"),
    gr.Slider(0, 500, step=1, label="Активность"),
    gr.Checkbox(label="Офлайн_участие")
]

# Заголовок и примеры
title = "Классификация увольнения"


# Создание интерфейса
demo = gr.Interface(
    fn=predict_exit,
    inputs=inputs,
    outputs="text",
    title=title,
    description="Введите параметры для предсказания увольнения",
)

demo.launch(inline=True, share=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [6]:
import gradio as gr
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load the model and label encoders
model = joblib.load('/rf_model.joblib') # Make sure this path is correct relative to where app.py will be run
le = joblib.load('/label_encoders.joblib') # Make sure this path is correct relative to where app.py will be run

# Define features and target
features=['Пол',
          'ОпытВДолжности',
          'Активность',
          'Оценка_HR',
          'УровеньОплаты',
          'Офлайн_участие',
          'Образование',
          'Город',
          'Закрытые_проекты',
          'ГодНайма',
          'Курсы']
target='Увольнение'
target_names = ['Не уволится', 'Уволится']
cat_features=['Закрытые_проекты',
    'Офлайн_участие'
             ]

def predict_exit(sex, experience, activity, hr_rating, pay_level, offline_p,
                 education, city, closed_projects, year_of_hire, courses):
    # СОЗДАЕМ DATAFRAME ИЗ ВХОДНЫХ ДАННЫХ
    input_data = pd.DataFrame([[sex, experience, activity, hr_rating, pay_level,
                                offline_p, education, city, closed_projects,
                                year_of_hire, courses]],
                              columns=features)

    # ГРУППИРОВАНИЕ ГОРОДОВ
    input_data['Город'] = input_data['Город'].replace({'Bangalore': 'Big_City',
                                                     'New Delhi': 'Big_City'})

    # ПОРЯДКОВОЕ КОДИРОВАНИЕ ОБРАЗОВАНИЯ
    edu_map = {'PHD': 0, 'Bachelors': 1, 'Masters': 2}
    input_data['Образование'] = input_data['Образование'].map(edu_map)

    # ПРИМЕНЕНИЕ LABEL ENCODER ДЛЯ КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ
    input_data['Пол'] = le['Пол'].transform(input_data['Пол'])
    input_data['Город'] = le['Город'].transform(input_data['Город'])

    # ПРЕДСКАЗАНИЕ
    prediction = model.predict(input_data)[0]
    result = 'Уволится' if prediction else 'Не уволится'

    # ВЕРОЯТНОСТИ
    proba = model.predict_proba(input_data)[:,prediction][0]

    return f"Вид: {result}\n Уверенность: {proba}"

# Описание входных полей
inputs = [
    gr.Dropdown(choices=list(le['Пол'].classes_), label="Пол"),
    gr.Slider(0, 7, step=1, label="ОпытВДолжности"),
    gr.Slider(0, 500, step=1, label="Активность"),
    gr.Slider(0, 10, step=1, label="Оценка_HR"),
    gr.Slider(1, 3, step=1, label="УровеньОплаты"),
    gr.Checkbox(label="Офлайн_участие"),
    gr.Dropdown(choices=['Bachelors', 'Masters', 'PHD'], label="Образование"),
    gr.Dropdown(choices=['Pune', 'New Delhi', 'Bangalore'], label="Город"),
    gr.Slider(0, 15, step=1, label="Закрытые_проекты"),
    gr.Slider(2012, 2018, step=1, label="ГодНайма"),
    gr.Slider(0, 5, step=1, label="Курсы")
]

# Заголовок и примеры
title = "Классификация увольнения"

# Custom CSS to enlarge the output window and font size and the overall interface window
custom_css = """
.gradio-container {
    min-height: 80vh !important; /* Make the container take up at least 80% of the viewport height */
    max-width: 90% !important;
    margin: auto !important;
}
.output-textarea {
    height: 250px !important;
    font-size: 1.4em !important;
}
"""

# Создание интерфейса
demo = gr.Interface(
    fn=predict_exit,
    inputs=inputs,
    outputs="text",
    title=title,
    description="Введите параметры для предсказания увольнения",
    css=custom_css
)

# Запуск (в Jupyter используйте inline=True)
demo.launch(inline=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ecc582d19926826e35.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
